# Romania (RO) Monthly Data Cleaning

**Pipeline**
```
Data_Original = read Sheet1            (frozen reference, never touched)
Data          = Data_Original.copy()   (working sheet - all cleaning here)
   ...cleaning steps...
Dataraw_df    = Data.copy()            (after-cleaning FINAL df -> Step 2)
```


In [1]:
import pandas as pd
import re

# ---- Parameters (dynamic / override here) ----
RAW_FILE   = "RO_06_2026__1_.xlsx"   # raw input, format 'RO MM YY' / 'RO MM YYYY'
DATA_SHEET = "Sheet1"                 # sheet holding the main data
PRIOR_FILE = None                     # last month's archived final, e.g. r"Archive\\Country_Dashboard_Data_-_Romania_Data_May26.xlsx"

print(f"Raw file : {RAW_FILE}")

Raw file : RO_06_2026__1_.xlsx


## Step 1 - Load Sheet1, freeze it as `Data_Original`, copy to `Data`

In [2]:
# Read Sheet1 (header in row 1) into the frozen reference DataFrame.
Data_Original = pd.read_excel(RAW_FILE, sheet_name=DATA_SHEET, header=0)

# Working copy - every cleaning change happens on `Data`. Data_Original stays untouched.
Data = Data_Original.copy()

print(f"Data_Original : {Data_Original.shape[0]:,} rows x {Data_Original.shape[1]} cols")
print(f"Data (copy)   : {Data.shape[0]:,} rows x {Data.shape[1]} cols")
print(f"Independent copy? {Data is not Data_Original}")
print()
print("First 6 columns (dimensions):", list(Data.columns[:6]))
print("First 3 month columns       :", list(Data.columns[6:9]))
print("Last 3 month columns        :", list(Data.columns[-3:]))

Data_Original : 26,696 rows x 120 cols
Data (copy)   : 26,696 rows x 120 cols
Independent copy? True

First 6 columns (dimensions): ['BRICK', 'MARKET', 'PRODUCT_NAME', 'CORPORATION', 'CHANNEL', 'MEASURE']
First 3 month columns       : ["'JAN.17'", "'FEB.17'", "'MAR.17'"]
Last 3 month columns        : ["'APR.26'", "'MAY.26'", "'JUN.26'"]


### Input QC - check the raw file before any processing

Flags extra / missing / renamed columns (expected 117 = 6 dimensions + months), non-month headers with their Excel column letter, duplicate or missing months, unexpected MEASURE values, non-numeric cells, and an empty latest month. Any **FAIL** stops the run here.

In [ ]:
import importlib, santen_qc
importlib.reload(santen_qc)

santen_qc.run_input_qc(RAW_FILE, country="Romania", sheet=DATA_SHEET, prior_final=PRIOR_FILE)

## Step 2 - Retain only the last N months of back data

Keep the 6 dimension columns + the **trailing `N_BACK_MONTHS`** month columns (ending at `LATEST_MONTH`), drop all earlier months. Then create the final `Dataraw_df`.


In [3]:
# ---- Parameters (override here) ----
N_BACK_MONTHS = 56      # number of trailing months of back data to retain
LATEST_MONTH  = None    # None -> auto-use the last month column; or override e.g. "JUN.26" / "'JUN.26'"

DIM_COLS   = list(Data.columns[:6])   # BRICK, MARKET, PRODUCT_NAME, CORPORATION, CHANNEL, MEASURE
month_cols = list(Data.columns[6:])   # all month columns, in file order (newest last)

def _norm(m):
    # normalise a month label for matching: drop wrapping apostrophes, upper-case
    return str(m).strip().strip("'").upper()

# Resolve the latest month index
if LATEST_MONTH is None:
    latest_idx = len(month_cols) - 1
else:
    target = _norm(LATEST_MONTH)
    hits = [i for i, m in enumerate(month_cols) if _norm(m) == target]
    if not hits:
        raise ValueError(f"LATEST_MONTH {LATEST_MONTH!r} not found among month columns.")
    latest_idx = hits[0]

start_idx = latest_idx - N_BACK_MONTHS + 1
if start_idx < 0:
    raise ValueError(
        f"Only {latest_idx + 1} months available up to the latest; cannot retain {N_BACK_MONTHS}."
    )

keep_month_cols = month_cols[start_idx: latest_idx + 1]

# Apply to the working DataFrame
Data = Data[DIM_COLS + keep_month_cols].copy()

print(f"Retained {len(keep_month_cols)} months: {keep_month_cols[0]} ... {keep_month_cols[-1]}")
print(f"Data now: {Data.shape[0]:,} rows x {Data.shape[1]} cols  ({len(DIM_COLS)} dims + {len(keep_month_cols)} months)")

Retained 56 months: 'NOV.21' ... 'JUN.26'
Data now: 26,696 rows x 62 cols  (6 dims + 56 months)


In [4]:
# ---- Validation (order-independent) ----
# 1) Record count unchanged vs the frozen original
rows_ok = len(Data) == len(Data_Original)
print(f"[1] Row count: Data={len(Data):,}  Data_Original={len(Data_Original):,}  -> {'OK' if rows_ok else 'MISMATCH'}")

# 2) Column-sum check: each retained month must have identical sum to the original (values untouched)
sum_mismatches = []
for m in keep_month_cols:
    s_new = pd.to_numeric(Data[m], errors='coerce').sum()
    s_org = pd.to_numeric(Data_Original[m], errors='coerce').sum()
    if abs(s_new - s_org) > 1e-6:
        sum_mismatches.append((m, s_new, s_org))
print(f"[2] Column-sum match on all {len(keep_month_cols)} retained months -> "
      f"{'OK' if not sum_mismatches else f'{len(sum_mismatches)} MISMATCH'}")

# 3) Dimension columns preserved (names + order + content)
dims_ok = list(Data[DIM_COLS].columns) == DIM_COLS and Data[DIM_COLS].equals(Data_Original[DIM_COLS])
print(f"[3] Dimension columns unchanged -> {'OK' if dims_ok else 'MISMATCH'}")

# 4) Confirm dropped months are gone and none of the kept ones are missing
dropped = [m for m in month_cols if m not in keep_month_cols]
print(f"[4] Dropped {len(dropped)} earlier months (e.g. {dropped[0]} ... {dropped[-1]})")

[1] Row count: Data=26,696  Data_Original=26,696  -> OK
[2] Column-sum match on all 56 retained months -> OK
[3] Dimension columns unchanged -> OK
[4] Dropped 58 earlier months (e.g. 'JAN.17' ... 'OCT.21')


## Create final after-cleaning DataFrame `Dataraw_df`

In [5]:
Dataraw_df = Data.copy()
print(f"Dataraw_df ready: {Dataraw_df.shape[0]:,} rows x {Dataraw_df.shape[1]} cols")
print("Columns:", list(Dataraw_df.columns[:6]), "...", list(Dataraw_df.columns[-3:]))

Dataraw_df ready: 26,696 rows x 62 cols
Columns: ['BRICK', 'MARKET', 'PRODUCT_NAME', 'CORPORATION', 'CHANNEL', 'MEASURE'] ... ["'APR.26'", "'MAY.26'", "'JUN.26'"]


## Step 3 - Replicate the 4 Power Query tables in pandas

Faithful pandas translations of the Advanced-Editor M code:
1. **Data** -> `working_rom_df` (built from `Dataraw_df`)
2. **Bricks** -> `bricks_df`   (Mapping_Romania.xlsx / Bricks)
3. **Category** -> `category_df` (Mapping_Romania.xlsx / Category)
4. **Month** -> `month_df`    (Mapping_Romania.xlsx / Month)


In [6]:
MAPPING_FILE = "Mapping_Romania.xlsx"
print('Mapping file:', MAPPING_FILE)

Mapping file: Mapping_Romania.xlsx


### Query 1 - Data -> `data_df`

In [7]:
# ---- Query 1: Data  ->  data_df ----
# M steps: PromoteHeaders (already done) -> rename month cols 7..62 to M1..M56 (by POSITION)
#          -> UnpivotOtherColumns(6 dims) -> Pivot MEASURE (sum) -> filter CHANNEL<>'CH4'
#          -> add Custom = PRODUCT_NAME & CORPORATION -> units/value numeric

DIMS6      = list(Dataraw_df.columns[:6])              # BRICK,MARKET,PRODUCT_NAME,CORPORATION,CHANNEL,MEASURE
DIM5       = [c for c in DIMS6 if c != 'MEASURE']      # the 5 kept as row keys after pivot
month_cols = list(Dataraw_df.columns[6:])             # 56 month cols, in order (oldest->newest)
rename_map = {c: f'M{i+1}' for i, c in enumerate(month_cols)}   # NOV.21->M1 ... JUN.26->M56
Mcols      = list(rename_map.values())

w = Dataraw_df.rename(columns=rename_map)

# Unpivot the M columns (keep the 6 dims incl. MEASURE)
melted = w.melt(id_vars=DIMS6, value_vars=Mcols, var_name='Attribute', value_name='Value')

# Pivot MEASURE into columns via sum (min_count=1 keeps all-blank groups as NaN, like List.Sum(null)=null)
piv = (melted.groupby(DIM5 + ['Attribute', 'MEASURE'], dropna=False)['Value']
             .sum(min_count=1)
             .unstack('MEASURE')
             .reset_index())
piv.columns.name = None

# Preserve units/value order as first-seen in source, then filter out CH4
measure_order = list(dict.fromkeys(Dataraw_df['MEASURE']))     # ['units','value']
piv = piv[DIM5 + ['Attribute'] + measure_order]
data_df = piv[piv['CHANNEL'] != 'CH4'].copy()

# Add Custom = PRODUCT_NAME & CORPORATION
data_df['Custom'] = data_df['PRODUCT_NAME'].astype(str) + data_df['CORPORATION'].astype(str)

# Ensure numeric measures
for c in measure_order:
    data_df[c] = pd.to_numeric(data_df[c], errors='coerce')

data_df = data_df.reset_index(drop=True)
print('data_df:', data_df.shape)
print('columns:', list(data_df.columns))
data_df.head(4)

data_df: (526512, 9)
columns: ['BRICK', 'MARKET', 'PRODUCT_NAME', 'CORPORATION', 'CHANNEL', 'Attribute', 'units', 'value', 'Custom']


,BRICK,MARKET,PRODUCT_NAME,CORPORATION,CHANNEL,Attribute,units,value,Custom
0,Alba,S01A0 ANTI-INFECTIVES-EYE,S01A0 ANTI-INFECTIVES-EYE,BAUSCH HEALTH*,Hospital,M1,NaN,NaN,S01A0 ANTI-INFECTIVES-EYEBAUSCH HEALTH*
1,Alba,S01A0 ANTI-INFECTIVES-EYE,S01A0 ANTI-INFECTIVES-EYE,BAUSCH HEALTH*,Hospital,M10,NaN,NaN,S01A0 ANTI-INFECTIVES-EYEBAUSCH HEALTH*
2,Alba,S01A0 ANTI-INFECTIVES-EYE,S01A0 ANTI-INFECTIVES-EYE,BAUSCH HEALTH*,Hospital,M11,10.0,69.3,S01A0 ANTI-INFECTIVES-EYEBAUSCH HEALTH*
3,Alba,S01A0 ANTI-INFECTIVES-EYE,S01A0 ANTI-INFECTIVES-EYE,BAUSCH HEALTH*,Hospital,M12,NaN,NaN,S01A0 ANTI-INFECTIVES-EYEBAUSCH HEALTH*


### Query 2 - Bricks -> `bricks_df`

In [8]:
# ---- Query 2: Bricks -> bricks_df ----
bricks_df = pd.read_excel(MAPPING_FILE, sheet_name='Bricks', header=0)
# M keeps a 5th (blank-header) column as 'Column5'; replicate the name for fidelity
bricks_df = bricks_df.rename(columns={c: 'Column5' for c in bricks_df.columns if str(c).startswith('Unnamed')})
for c in ['Brick','Region','Territory','RSM']:
    bricks_df[c] = bricks_df[c].astype('string')
print('bricks_df:', bricks_df.shape, '| columns:', list(bricks_df.columns))
bricks_df.head(3)

bricks_df: (47, 4) | columns: ['Brick', 'Region', 'Territory', 'RSM']


,Brick,Region,Territory,RSM
0,ALBA,ARDEAL,Flaviu Rus,-
1,ARAD,BANAT,Ramona Arancutean,-
2,ARGES,BUH+OLTENIA,Rebeca Stavarache,-


### Query 3 - Category -> `category_df`

In [9]:
# ---- Query 3: Category -> category_df ----
category_df = pd.read_excel(MAPPING_FILE, sheet_name='Category', header=0)
# Add Custom = Product & Corporation  (join key back to working_rom_df.Custom)
category_df['Custom'] = category_df['Product'].astype(str) + category_df['Corporation'].astype(str)
print('category_df:', category_df.shape)
print('columns:', list(category_df.columns))
category_df.head(3)

category_df: (193, 17)
columns: ['Product', 'Brand', 'Corporation', 'Company', 'Pack original', 'International pack', 'Pack', 'HU conversion factor', 'ATC class', 'Category', 'Sub-market', 'Formulation original', 'Formulation', 'Molecule', 'Ingredient', 'Parallel import', 'Custom']


,Product,Brand,Corporation,Company,Pack original,International pack,Pack,HU conversion factor,ATC class,Category,Sub-market,Formulation original,Formulation,Molecule,Ingredient,Parallel import,Custom
0,FITOSTILL EYE DROPS 10 .5ML,FITOSTILL,ABOCA,ABOCA,FITOSTILL EYE DROPS 10 .5ML,FITOSTILL EYE DROPS 10 .5ML,EYE DROPS 10 .5ML,1,-,Artificial Tears,-,-,Eye drops - UD,-,HA,NO,FITOSTILL EYE DROPS 10 .5MLABOCA
1,MIRTILLO CAPS 70,MIRTILLO,ABOCA,ABOCA,MIRTILLO CAPS 70,CAPS 70,CAPS 70,1,-,Others,-,-,-,-,-,-,MIRTILLO CAPS 70ABOCA
2,SYSTANE BALANCE EYE DROPS 10ML,SYSTANE,ALCON*,ALCON,SYSTANE BALANCE EYE DROPS 10ML,SYSTANE BALANCE EYE DROPS 10ML,BALANCE EYE DROPS 10ML,1,-,Artificial Tears,-,-,Eye drops - MD,MACROGOL+PROPYLENE GLYCOL+HYDROXYPROPYL GUAR,Others,NO,SYSTANE BALANCE EYE DROPS 10MLALCON*


### Query 4 - Month -> `month_df`

In [10]:
# ---- Query 4: Month -> month_df ----
month_df = pd.read_excel(MAPPING_FILE, sheet_name='Month', header=0)
month_df['M']    = month_df['M'].astype('string')
month_df['Date'] = pd.to_datetime(month_df['Date'])
print('month_df:', month_df.shape, '| M1 ->', month_df['Date'].iloc[0].date(), '| M56 ->', month_df['Date'].iloc[-1].date())
month_df.head(3)

month_df: (56, 2) | M1 -> 2021-11-01 | M56 -> 2026-06-01


,M,Date
0,M1,2021-11-01
1,M2,2021-12-01
2,M3,2022-01-01


### Validation

In [11]:
# ---- Validation: data_df vs Dataraw_df (order-independent) ----
# [1] Column-sum: units/value totals must equal Dataraw_df month-sums for that MEASURE, excluding CH4
ref = Dataraw_df[Dataraw_df['CHANNEL'] != 'CH4']
ref_units = pd.to_numeric(ref[ref['MEASURE']=='units'][month_cols].stack(), errors='coerce').sum()
ref_value = pd.to_numeric(ref[ref['MEASURE']=='value'][month_cols].stack(), errors='coerce').sum()
got_units = data_df['units'].sum()
got_value = data_df['value'].sum()
print(f"[1] units sum: got={got_units:,.2f} ref={ref_units:,.2f} -> {'OK' if abs(got_units-ref_units)<1e-3 else 'MISMATCH'}")
print(f"    value sum: got={got_value:,.2f} ref={ref_value:,.2f} -> {'OK' if abs(got_value-ref_value)<1e-3 else 'MISMATCH'}")

# [2] Record count: unique non-CH4 5-dim combos x 56 months
exp_rows = ref[DIM5].drop_duplicates().shape[0] * len(Mcols)
print(f"[2] Row count: got={len(data_df):,} expected={exp_rows:,} -> {'OK' if len(data_df)==exp_rows else 'MISMATCH'}")

# [3] CH4 fully removed
print(f"[3] CH4 rows remaining: {(data_df['CHANNEL']=='CH4').sum()} -> {'OK' if (data_df['CHANNEL']=='CH4').sum()==0 else 'FAIL'}")

[1] units sum: got=43,241,379.00 ref=43,241,379.00 -> OK
    value sum: got=1,226,498,627.57 ref=1,226,498,627.57 -> OK
[2] Row count: got=526,512 expected=526,512 -> OK
[3] CH4 rows remaining: 0 -> OK


### Note
`data_df` is the melted **Data** query (intermediate). The enriched, Working-matching table is built next as `working_rom_df`.

In [12]:
print('data_df (Data query, intermediate):', data_df.shape)
print('Feeds Step 4 along with bricks_df / category_df / month_df.')

data_df (Data query, intermediate): (526512, 9)
Feeds Step 4 along with bricks_df / category_df / month_df.


## Step 4 - Build `working_rom_df` to match the Working file

Reverse-engineered from Working.xlsx. Pipeline:
1. Map `Attribute` (M1..M56) -> real `Date` via `month_df`.
2. **HU-convert units**: `units_conv = units x HU conversion factor` (Category, on `Custom`). `value` unchanged.
3. **Aggregate across CHANNEL** (Pharmacy+Hospital) per (PRODUCT_NAME,MARKET,BRICK,CORPORATION,Custom,Date).
4. Keep the **last `WINDOW_MONTHS` (=24)** months (dynamic, ends at latest month).
5. Merge **Bricks** (Region,RSM,Territory) on BRICK (**case-insensitive** join) and **Category** descriptive fields on `Custom`.
6. Pivot to wide: **units block** then **value block** (24 date cols each).
7. Drop combos with no data in the window.

> **FLAG:** `Retina Market` is `'Exclude'` for every row in the reference and its source isn't in any file provided, so it is set to the constant `'Exclude'` pending the real rule.

In [13]:
WINDOW_MONTHS = 24   # trailing months shown in the Working layout (override here)

# Retina Market rule: 'Include' if the trigger word appears in the Category column, else 'Exclude'.
RETINA_INCLUDE_WORD   = 'retina'    # 'retina' in Category => Include, else Exclude
RETINA_MATCH          = 'contains'  # 'contains' (substring) or 'exact'
RETINA_CASE_SENSITIVE = False

KEYS = ['PRODUCT_NAME','MARKET','BRICK','CORPORATION','Custom']
DESC_ORDER = ['PRODUCT_NAME','MARKET','Retina Market','Region','RSM','Territory','BRICK',
              'Category','Brand','Company','Pack','Molecule','Sub Market','Formulation','Ingredient','Import']

# 1) date map + 2) HU conversion on units
huf = dict(zip(category_df['Custom'], pd.to_numeric(category_df['HU conversion factor'], errors='coerce')))
w = data_df.merge(month_df[['M','Date']], left_on='Attribute', right_on='M', how='left')
w['units_conv'] = pd.to_numeric(w['units'], errors='coerce') * w['Custom'].map(huf)
w['value_num']  = pd.to_numeric(w['value'], errors='coerce')

# 3) aggregate across CHANNEL
agg_u = w.groupby(KEYS+['Date'])['units_conv'].sum(min_count=1)
agg_v = w.groupby(KEYS+['Date'])['value_num'].sum(min_count=1)
agg = pd.concat([agg_u.rename('units'), agg_v.rename('value')], axis=1).reset_index()

# 4) trailing window
window = sorted(agg['Date'].dropna().unique())[-WINDOW_MONTHS:]
sub = agg[agg['Date'].isin(window)]

# 6) pivot to wide blocks, aligned to one row order
row_keys = sub[KEYS].drop_duplicates().reset_index(drop=True)
idx = pd.MultiIndex.from_frame(row_keys)
uu = sub.pivot(index=KEYS, columns='Date', values='units').reindex(index=idx, columns=window)
vv = sub.pivot(index=KEYS, columns='Date', values='value').reindex(index=idx, columns=window)

# 5) descriptive: Bricks (case-insensitive) + Category (on Custom)
D = row_keys.copy()
bmap = bricks_df.assign(BKEY=bricks_df['Brick'].astype(str).str.upper())[['BKEY','Region','Territory','RSM']]
D['BKEY'] = D['BRICK'].astype(str).str.upper()
D = D.merge(bmap, on='BKEY', how='left').drop(columns='BKEY')
cmap = category_df.rename(columns={'Sub-market':'Sub Market','Parallel import':'Import'})[
        ['Custom','Category','Brand','Company','Pack','Molecule','Sub Market','Formulation','Ingredient','Import']]
D = D.merge(cmap, on='Custom', how='left')

# Retina Market from Category keyword
cat_col = D['Category'].astype('string').fillna('')
if RETINA_INCLUDE_WORD:
    probe = cat_col if RETINA_CASE_SENSITIVE else cat_col.str.lower()
    word  = RETINA_INCLUDE_WORD if RETINA_CASE_SENSITIVE else RETINA_INCLUDE_WORD.lower()
    hit = probe.str.contains(re.escape(word)) if RETINA_MATCH=='contains' else probe.eq(word)
    D['Retina Market'] = hit.map({True:'Include', False:'Exclude'})
else:
    D['Retina Market'] = 'Exclude'

# assemble flat table (duplicate dd-mm-yyyy headers: units block then value block)
date_labels = [pd.Timestamp(d).strftime('%d-%m-%Y') for d in window]
desc = D[DESC_ORDER].reset_index(drop=True)
units_block = uu.reset_index(drop=True); units_block.columns = date_labels
value_block = vv.reset_index(drop=True); value_block.columns = date_labels
result = pd.concat([desc, units_block, value_block], axis=1)

# 7) drop combos with no data in the window
mask = pd.concat([units_block, value_block], axis=1).abs().sum(axis=1, skipna=True) > 0
working_rom_df = result[mask.values].reset_index(drop=True)

print('working_rom_df:', working_rom_df.shape)
print('date block:', date_labels[0], '...', date_labels[-1], '(units block then value block)')
print('Retina Market:', working_rom_df['Retina Market'].value_counts().to_dict())

working_rom_df: (4678, 64)
date block: 01-07-2024 ... 01-06-2026 (units block then value block)
Retina Market: {'Exclude': 4678}


### Validation vs Working.xlsx

In [14]:
# ---- Validation vs Working.xlsx (order-independent) ----
import numpy as np
wref = pd.read_excel('Working.xlsx', sheet_name='Data', header=2)
print(f"[1] Rows: got={len(working_rom_df):,}  Working={len(wref):,}  -> {'OK' if len(working_rom_df)==len(wref) else 'MISMATCH'}")
desc_ok = list(working_rom_df.columns[:16]) == list(wref.columns[:16])
print(f"[2] Descriptive headers match Working -> {'OK' if desc_ok else 'MISMATCH'}")
g = np.array([pd.to_numeric(working_rom_df.iloc[:,16+i], errors='coerce').sum() for i in range(48)])
r = np.array([pd.to_numeric(wref.iloc[:,16+i], errors='coerce').sum() for i in range(48)])
maxdiff = np.nanmax(np.abs(g-r))
print(f"[3] All 48 date-column sums match Working -> {'OK' if maxdiff < 1e-3 else f'MISMATCH (max {maxdiff:.4f})'}")
print(f"    units {working_rom_df.columns[16]}: got={g[0]:,.2f} ref={r[0]:,.2f}")
print(f"    value {working_rom_df.columns[40]}: got={g[24]:,.2f} ref={r[24]:,.2f}")

[1] Rows: got=4,678  Working=4,678  -> OK
[2] Descriptive headers match Working -> OK
[3] All 48 date-column sums match Working -> OK
    units 01-07-2024: got=890,574.00 ref=890,574.00
    value 01-07-2024: got=25,743,720.94 ref=25,743,720.94


### Output

In [15]:
# Output with a 2-row header like the Working pivot (value-field label over dd-mm-yyyy dates)
top = ['']*16 + ['Sum of HU Conv Units']*len(date_labels) + ['Sum of value']*len(date_labels)
bot = DESC_ORDER + date_labels + date_labels
header_df = pd.DataFrame([top, bot])
body = pd.DataFrame(working_rom_df.to_numpy())
out = pd.concat([header_df, body], ignore_index=True)
out.to_excel('working_rom_df.xlsx', sheet_name='Data', header=False, index=False)
print('Wrote working_rom_df.xlsx  (2 header rows +', len(working_rom_df), 'data rows)')

Wrote working_rom_df.xlsx  (2 header rows + 4678 data rows)


## Step 5 - New-product check (mapping coverage)

After the Category mapping is merged, a blank in **Brand / Company / Category** means the product's `Custom` key isn't in `Mapping_Romania.xlsx` -> a **new product** to add to the mapping. This only reports; it doesn't auto-update.

In [16]:
CHECK_COLS = ['Brand','Company','Category']

chk = working_rom_df[CHECK_COLS].astype('string')
blank_mask = chk.isna().any(axis=1) | chk.apply(lambda s: s.str.strip().eq('')).any(axis=1)
n_blank = int(blank_mask.sum())

if n_blank > 0:
    print('New product detected')
    new_products = (working_rom_df.loc[blank_mask, ['PRODUCT_NAME','MARKET','BRICK']]
                    .drop_duplicates().reset_index(drop=True))
    print(f'{blank_mask.sum()} rows / {new_products["PRODUCT_NAME"].nunique()} product(s) missing from the mapping:')
    for p in sorted(new_products['PRODUCT_NAME'].unique()):
        print('   -', p)
else:
    print('no new products found')

no new products found


## Step 6 - Build the Country Dashboard `Data` table (with metrics)

Assemble the 106-column dashboard sheet from `working_rom_df`:
`Key` + descriptive (B-O) + 24 `UN-*` + 24 `REV-*` + 42 computed metrics (BL-DA) + `Key 2`.
Fiscal year starts **April**. Metrics validated to reproduce the reference dashboard exactly.

In [17]:
import numpy as np

DASH_FILE = 'Country_Dashboard_Data_-_Romania.xlsx'

# --- source blocks from working_rom_df (positional: 0-15 desc, 16-39 units, 40-63 value) ---
desc_src = working_rom_df.iloc[:, 0:16].copy()
UN  = working_rom_df.iloc[:, 16:40].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()
REV = working_rom_df.iloc[:, 40:64].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()
win_dates = [pd.Timestamp('20'+d[6:]+'-'+d[3:5]+'-'+d[0:2]) if False else pd.to_datetime(d, format='%d-%m-%Y')
            for d in date_labels]   # 24 month timestamps Jul24..Jun26

# --- fiscal helpers (FY starts April) ---
def fq(m):  return {4:1,5:2,6:3,7:1,8:2,9:3,10:1,11:2,12:3,1:1,2:2,3:3}[m]   # months since fiscal-qtr start
def ft(m):  return {4:1,5:2,6:3,7:4,8:1,9:2,10:3,11:4,12:1,1:2,2:3,3:4}[m]   # months since fiscal-tercile start
def fy(m):  return m-3 if m>=4 else 9+m                                       # months since fiscal April
SPAN = {'':1,'Fixed Qtr ':None,'Rolling Qtr ':3,'Fixed Tercile ':None,'Rolling Tercile ':4,'FY YTD ':None,'MAT ':12}
ORDER = ['','Fixed Qtr ','Rolling Qtr ','Fixed Tercile ','Rolling Tercile ','FY YTD ','MAT ']

def span_for(tag, r):
    m = win_dates[r].month
    if tag=='Fixed Qtr ':     return fq(m)
    if tag=='Fixed Tercile ': return ft(m)
    if tag=='FY YTD ':        return fy(m)
    return SPAN[tag]

def metric(block, r, tag):
    c = span_for(tag, r); s = r-c+1
    return block[:, s:r+1].sum(axis=1)

# --- reference months: Current, Previous Period, Previous Year ---
n = len(win_dates)
refs = {'Current': n-1, 'Previous Period': n-2, 'Previous Year': n-13}

metric_cols, metric_names = [], []
for meas, block in [('UN',UN),('REV',REV)]:
    for period, r in refs.items():
        lbl = win_dates[r].strftime('%b %Y')
        for tag in ORDER:
            metric_cols.append(metric(block, r, tag))
            metric_names.append(f'{period} {meas}-{tag}{lbl}')
metrics = np.column_stack(metric_cols)

# --- descriptive rename to dashboard headers ---
desc = desc_src.rename(columns={'Retina Market':'Retina market','BRICK':'Brick','Sub Market':'Sub-Market'})
DESC_DASH = ['Retina market','Region','RSM','Territory','Brick','Category','Brand','Company',
             'Pack','Molecule','Sub-Market','Formulation','Ingredient','Import']
desc = desc[DESC_DASH].reset_index(drop=True)

# --- Key (=Brand&Formulation&Company) and Key 2 (=Brick&Territory), as computed text ---
key  = desc['Brand'].astype(str)+desc['Formulation'].astype(str)+desc['Company'].astype(str)
key2 = desc['Brick'].astype(str)+desc['Territory'].astype(str)

# --- month header names ---
un_names  = ['UN-'+d.strftime('%b %Y')  for d in win_dates]
rev_names = ['REV-'+d.strftime('%b %Y') for d in win_dates]

# --- assemble 106-col dashboard Data table ---
dashboard_data_df = pd.concat([
    pd.Series(key, name='Key'),
    desc,
    pd.DataFrame(UN,  columns=un_names),
    pd.DataFrame(REV, columns=rev_names),
    pd.DataFrame(metrics, columns=metric_names),
    pd.Series(key2, name='Key 2'),
], axis=1)

print('dashboard_data_df:', dashboard_data_df.shape)
print('first cols:', list(dashboard_data_df.columns[:4]))
print('metric sample:', metric_names[:3], '...', metric_names[-2:])
print('last col:', dashboard_data_df.columns[-1])

dashboard_data_df: (4678, 106)
first cols: ['Key', 'Retina market', 'Region', 'RSM']
metric sample: ['Current UN-Jun 2026', 'Current UN-Fixed Qtr Jun 2026', 'Current UN-Rolling Qtr Jun 2026'] ... ['Previous Year REV-FY YTD Jun 2025', 'Previous Year REV-MAT Jun 2025']
last col: Key 2


### Validation

In [18]:
# ---- Validation: structure vs dashboard, and internal consistency ----
ref_cols = pd.read_excel(DASH_FILE, sheet_name='Data', header=0).columns.tolist()
got_cols = dashboard_data_df.columns.tolist()
print(f"[1] 106 columns, names+order match dashboard -> {'OK' if got_cols==ref_cols else 'MISMATCH'}")
if got_cols!=ref_cols:
    for i,(a,b) in enumerate(zip(got_cols,ref_cols)):
        if a!=b: print('   diff at',i,':',a,'vs',b)

# single-month metric must equal that month's raw column
c_un  = np.allclose(dashboard_data_df['Current UN-'+win_dates[-1].strftime('%b %Y')], UN[:,-1])
c_rev = np.allclose(dashboard_data_df['Current REV-'+win_dates[-1].strftime('%b %Y')], REV[:,-1])
print(f"[2] Current single-month == raw month column (UN & REV) -> {'OK' if c_un and c_rev else 'MISMATCH'}")
# MAT current == sum of all 12 trailing incl current
mat_ok = np.allclose(dashboard_data_df['Current UN-MAT '+win_dates[-1].strftime('%b %Y')], UN[:,-12:].sum(axis=1))
print(f"[3] Current UN-MAT == sum last 12 months -> {'OK' if mat_ok else 'MISMATCH'}")

[1] 106 columns, names+order match dashboard -> OK
[2] Current single-month == raw month column (UN & REV) -> OK
[3] Current UN-MAT == sum last 12 months -> OK


## Step 7 - Write the `Data` sheet (formatting preserved)

Output a single-sheet file whose `Data` sheet reuses the **original header** (text, colors) and **column widths**, with the pandas-computed June table written beneath. `Key` (A) and `Key 2` (DB) are left **empty** (handled manually).
> openpyxl is used only to carry the original formatting; all values are pandas-computed.

In [19]:
import openpyxl
OUT_FILE = 'Country_Dashboard_Data_-_Romania_Data.xlsx'

# start from the original workbook so header formatting + column widths are inherited
wb = openpyxl.load_workbook(DASH_FILE)
for s in list(wb.sheetnames):
    if s != 'Data':
        del wb[s]
ws = wb['Data']

# write data beneath the existing header (row 1 untouched -> keeps colors/text)
# df col order == sheet col order; skip Key (df 0 / col A) and Key 2 (df 105 / col DB)
vals = dashboard_data_df.to_numpy(dtype=object)
n_new = len(dashboard_data_df)
for r in range(n_new):
    er = r + 2
    for cidx in range(1, 105):                 # df cols 1..104 -> sheet cols 2..105
        v = vals[r][cidx]
        ws.cell(row=er, column=cidx+1).value = (None if pd.isna(v) else v)
    ws.cell(row=er, column=1).value   = None   # Key   (A) empty
    ws.cell(row=er, column=106).value = None   # Key 2 (DB) empty

# clear any leftover rows from the original (old had 1 more row than new)
for er in range(n_new + 2, ws.max_row + 1):
    for c in range(1, 107):
        ws.cell(row=er, column=c).value = None

wb.save(OUT_FILE)
chk = openpyxl.load_workbook(OUT_FILE, read_only=True)
print('Wrote', OUT_FILE, '| sheets:', chk.sheetnames, '| data rows:', n_new)

Wrote Country_Dashboard_Data_-_Romania_Data.xlsx | sheets: ['Data'] | data rows: 4678


## Step 8 - QC checks on the saved output

Runs the shared `santen_qc.py` checks against `OUT_FILE` as saved on disk: structure, formatting, data integrity, metric recomputation, totals vs the raw file, anomalies, and comparison with last month's final.

Any **FAIL** stops here with an error. A report `<OUT_FILE>_QC.xlsx` is written next to the output.

> `santen_qc.py` must be in the same folder as this notebook. Set `PRIOR_FILE` (parameters cell) to last month's archived final (this notebook overwrites `OUT_FILE` each run, so keep a copy before running).

In [ ]:
import importlib, santen_qc
importlib.reload(santen_qc)   # picks up edits to santen_qc.py without restarting

qc = santen_qc.run_qc(
    OUT_FILE,
    country="Romania",
    prior_file=PRIOR_FILE,
    source={"raw_file": RAW_FILE, "mapping_file": MAPPING_FILE, "sheet": DATA_SHEET},
)